In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [4]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [5]:
control_key = "is_control"
condition_rep_keys = "perturbation_embeddings"
condition_combined_keys = "condition_combined"
mass_deduct_keys = "bc1_well" # or None
random_seed = 42

condition_keys = "cytokine" # 数据集perturbation所对应的obs列名
dataset_name = "PBMC_all_hvg2000_gemini_test1"
sample_rep = "X_pca" #"X_scVI"  "X_flatvi" "X_state"

cov_config = {
    "donor": {
        "type": "categorical",
        "control_ot": "groupwise",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_source": "both",
        "contain_in_condition": True,
        "condition_source": "both",
    },
    # "cell_type": {
    #     "type": "categorical",
    #     "control_ot": "global",
    #     "perturbed_ot": "global",
    #     "use_in_model": True,
    #     "model_source": "control",
    #     "contain_in_condition": False,
    #     "condition_source": None,
    # },
}


if_adata_ref = None #用于根据一个参考adata快速构建pca
adata_ref_path = "data/processed/PBMC2000_pca_rep_0.2_42.h5ad"


#condition_rep_dict = pd.read_pickle("./data/processed/PBMC_cytokines.pkl")
condition_rep_dict = pd.read_pickle("./data/processed/condition_embedding_cytokine_PBMC_gemini_test1.pkl")
condition_rep_dict = {
    k: (v["embedding"] if isinstance(v, dict) and "embedding" in v else None)
    for k, v in condition_rep_dict.items()
}

In [39]:
filePath = './data/raw/PBMC_all_hvg2000.h5ad'
adata = sc.read_h5ad(filePath)
adata.uns["cov_config"] = cov_config
print(adata)

AnnData object with n_obs × n_vars = 9697974 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'donor_one_hot'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config'
    layers: 'counts'


In [40]:
adata.obs[control_key] = (adata.obs[condition_keys] == "PBS")
condition_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    9068273
True      629701
Name: count, dtype: int64


In [41]:
del adata.layers["counts"]

## splitting

In [30]:
#adata = adata[~adata.obs[condition_keys].isin(["LT-alpha2-beta1","IFN-lambda2","IFN-lambda3","IL-18Ra","LT-alpha1-beta2"])]

In [42]:
rng = np.random.default_rng(random_seed)
test_ratio = "cellflow"
donor_num = 8
condition_list = list(condition_list)
zero_shot = True

# 建议先统一 condition 表示，避免 condition_keys 是多列时 zeroshot 分支出错
if isinstance(condition_keys, str):
    condition_series = adata.obs[condition_keys].astype(str)
else:
    condition_series = adata.obs[condition_keys].astype(str).agg("_".join, axis=1)

if not zero_shot:
    # 分层抽样，先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = condition_series.loc[pert_mask].values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()

    adata_train.uns["normalized_m"] = 1 / (1 - test_ratio)
    adata_test.uns["normalized_m"] = 1 / test_ratio
    adata_control.uns["normalized_m"] = 1

else:
    # --------------------------------------------------
    # zero-shot: 按 condition 划 test
    # 但从 donor 中随机抽 donor_num 个，
    # 把这些 donor 在 test_condition 下的样本保留到训练集
    # --------------------------------------------------
    if test_ratio == "cellflow":
        test_condition = ["4-1BBL", "ADSF", "APRIL", "BAFF", "C5a", "IFN-beta", "IL-13", "IL-15", "Noggin", "OSM", "OX40L", "IFN-epsilon"]
    else:
        n_test = max(1, int(len(condition_list) * test_ratio))
        test_condition = rng.choice(condition_list, size=n_test, replace=False).tolist()
    train_condition = [c for c in condition_list if c not in test_condition]

    # donor 抽样
    all_donors = sorted(adata.obs["donor"].dropna().unique().tolist())
    donor_num = min(donor_num, len(all_donors))
    selected_donors = rng.choice(all_donors, size=donor_num, replace=False).tolist()

    print("test_condition:", test_condition)
    print("train_condition:", train_condition)
    print("selected_donors_for_test_in_train:", selected_donors)

    control_mask = adata.obs[control_key] == True
    pert_mask = ~control_mask

    train_condition_mask = condition_series.isin(train_condition)
    test_condition_mask = condition_series.isin(test_condition)
    selected_donor_mask = adata.obs["donor"].isin(selected_donors)

    # 训练集：
    # 1) 所有 train_condition
    # 2) test_condition 里属于 selected_donors 的样本
    train_mask = pert_mask & (
        train_condition_mask |
        (test_condition_mask & selected_donor_mask)
    )

    # 测试集：
    # test_condition 里不属于 selected_donors 的样本
    test_mask = pert_mask & test_condition_mask & (~selected_donor_mask)

    adata_control = adata[control_mask].copy()
    adata_train = adata[train_mask].copy()
    adata_test = adata[test_mask].copy()

    adata_train.uns["normalized_m"] = 1
    adata_test.uns["normalized_m"] = 1
    adata_control.uns["normalized_m"] = 1

    print("train cells:", adata_train.n_obs)
    print("test cells:", adata_test.n_obs)
    print("control cells:", adata_control.n_obs)


test_condition: ['4-1BBL', 'ADSF', 'APRIL', 'BAFF', 'C5a', 'IFN-beta', 'IL-13', 'IL-15', 'Noggin', 'OSM', 'OX40L', 'IFN-epsilon']
train_condition: ['Megalin', 'CD27L', 'CD30L', 'CD40L', 'CT-1', 'Decorin', 'EGF', 'EPO', 'FGF-beta', 'FLT3L', 'FasL', 'G-CSF', 'GDNF', 'GITRL', 'GM-CSF', 'HGF', 'IFN-alpha1', 'IFN-gamma', 'IFN-lambda1', 'IFN-lambda2', 'IFN-lambda3', 'IFN-omega', 'IGF-1', 'IL-1-alpha', 'IL-1-beta', 'IL-10', 'IL-11', 'IL-12', 'IL-16', 'IL-17A', 'IL-17B', 'IL-17C', 'IL-17D', 'IL-17E', 'IL-17F', 'IL-18Ra', 'IL-19', 'IL-1Ra', 'IL-2', 'IL-20', 'IL-21', 'IL-22', 'IL-23', 'IL-24', 'IL-26', 'IL-27', 'IL-3', 'IL-31', 'IL-32-beta', 'IL-33', 'IL-34', 'IL-35', 'IL-36-alpha', 'IL-36Ra', 'IL-4', 'IL-5', 'IL-6', 'IL-7', 'IL-8', 'IL-9', 'LIF', 'LIGHT', 'LT-alpha1-beta2', 'LT-alpha2-beta1', 'Leptin', 'M-CSF', 'PRL', 'PSPN', 'RANKL', 'SCF', 'LAP-TGF-beta1', 'TL1A', 'TNF-alpha', 'TPO', 'TRAIL', 'TSLP', 'TWEAK', 'VEGF']
selected_donors_for_test_in_train: ['Donor8', 'Donor9', 'Donor7', 'Donor4', 

In [43]:
del adata

## latent embedding

In [44]:
n_comps = 100
n_hidden = 1024
n_layers = 2

model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [45]:
if if_adata_ref:
    adata_ref = sc.read_h5ad(adata_ref_path,backed='r')
else:
    adata_ref = None

In [46]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_ref = adata_ref,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    condition_combined_keys = condition_combined_keys,
    cov_config = cov_config,
    condition_rep_dict = condition_rep_dict,
    pca_method = "scanpy", # "parse"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

[8.019994   4.294682   3.586406   2.5239506  2.0650702  1.8600937
 1.8149986  1.750735   1.5911564  1.5682368  1.456494   1.3736937
 1.2929538  1.2241945  1.2195226  1.1851397  1.1451101  1.127547
 1.099995   1.079837   1.0553141  1.0620013  1.0493817  1.0142832
 1.0015056  0.9921923  0.9765842  0.9682123  0.9573394  0.95015365
 0.9533835  0.936676   0.9219262  0.9195513  0.91910464 0.910123
 0.8980374  0.89363444 0.88889563 0.88223773 0.8757408  0.8675976
 0.86285365 0.85773927 0.853165   0.84415483 0.8204822  0.832345
 0.821401   0.81770235 0.81409794 0.81563604 0.7943719  0.79298335
 0.7920084  0.7924085  0.7754093  0.7773616  0.7687083  0.7704426
 0.7709598  0.761194   0.75662446 0.7527645  0.7513458  0.74846375
 0.7485515  0.74077183 0.7363773  0.7346762  0.7328195  0.7271187
 0.72640425 0.726765   0.71635836 0.719041   0.7144044  0.7076429
 0.7066291  0.69649404 0.6991027  0.6981275  0.69311124 0.68719363
 0.6918504  0.6897298  0.68814427 0.68127066 0.67738116 0.6710602
 0.672709

In [47]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{donor_num}_{zero_shot}_{sample_rep}_{n_comps}_{if_adata_ref}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [48]:
print(preprocess_save_path)
print(adata_control)
print(adata_train)
print(adata_test)

./data/processed/PBMC_all_hvg2000_gemini_test1_42_cellflow_8_True_X_pca_100_None
AnnData object with n_obs × n_vars = 629701 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'donor_one_hot', 'is_control', 'donor_idx', 'condition_combined'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config', 'normalized_m', 'pca', 'perturbation_embeddings', 'global_rulebook'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
AnnData object with n_obs × n_vars = 8695794 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pc

In [38]:
adata_control.uns

{'hvg': {'flavor': 'seurat'},
 'log1p': {},
 'cov_config': {'donor': {'type': 'categorical',
   'control_ot': 'groupwise',
   'perturbed_ot': 'groupwise',
   'use_in_model': True,
   'model_source': 'both',
   'contain_in_condition': True,
   'condition_source': 'both'}},
 'normalized_m': 1,
 'pca': {'params': {'zero_center': False,
   'use_highly_variable': False,
   'mask_var': None,
   'layer': 'X_centered'},
  'variance': array([60.0748    , 17.262188  , 13.15141   ,  6.309972  ,  4.0548916 ,
          3.4847312 ,  3.1374037 ,  3.0069945 ,  2.5463622 ,  2.3694782 ,
          2.0970423 ,  1.9659998 ,  1.692516  ,  1.5233266 ,  1.5058374 ,
          1.398755  ,  1.3009559 ,  1.2722096 ,  1.2034439 ,  1.1867514 ,
          1.120926  ,  1.1093596 ,  1.0635418 ,  1.0325078 ,  0.99872154,
          0.99155194,  0.9540602 ,  0.9428741 ,  0.9200804 ,  0.9028753 ,
          0.8889075 ,  0.87896025,  0.86255157,  0.8503403 ,  0.8413273 ,
          0.81694204,  0.8057262 ,  0.80039454,  0.796

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()